# cycle-detection-temp-set — ex2: return the cycle PATH (list of nodes around the loop) for debugging

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cycle-detection-temp-set`. Running the final beacon cell reports progress against the `Backprop: cycle detection via temp set` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cycle PATH detection — quick refresher

A boolean `has_cycle` tells you THAT a cycle exists. For debugging (or for producing actionable error messages), you usually want the ACTUAL nodes on the cycle. The same temp-set DFS can capture them:

```python
def find_cycle(root, get_children):
    path = []          # nodes currently on the DFS stack, in order
    on_stack = set()   # ids of those nodes (O(1) lookup)
    perm = set()

    def visit(node):
        nid = id(node)
        if nid in perm: return None
        if nid in on_stack:
            # back-edge — slice path from first appearance + this node
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            return path[i:] + [node]
        on_stack.add(nid); path.append(node)
        for child in get_children(node):
            cyc = visit(child)
            if cyc is not None: return cyc
        on_stack.discard(nid); path.pop()
        perm.add(nid)
        return None
    return visit(root)
```

Returns `None` for a DAG; for a cycle, returns the list of nodes around the loop (closing node repeated at the end). The cost is one extra `path` list — `O(depth)` memory — and a `next()` slice once per cycle hit.

### Exercise 2 — return the cycle PATH (list of nodes around the loop) for debugging

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the temp-set DFS cycle-detection pattern to RETURN the actual list of nodes around the cycle (instead of just bool), by slicing the in-flight path from the first appearance of the back-edge target.
> Keywords: cycle-detection, cycle-path, debug, back-edge, temp-set
> ```

**KCs targeted:** `cycle-detection-temp-set`, `dfs-three-set-toposort`

Implement `find_cycle(root, get_children)` — like `has_cycle` from ex1, but instead of returning `True/False`, return either:
- `None` — no cycle reachable from `root` (the DAG case), OR
- A `list[node]` — the nodes around the cycle, in DFS-traversal order, with the back-edge target REPEATED at the end so the cycle is closed.

**Why this is useful.** A boolean tells you 'something is wrong'. The actual cycle path lets you write an error message like `Cycle: a -> b -> c -> a` — which the user can fix.

**Algorithm sketch.** Same temp-set DFS as ex1, plus an auxiliary ORDERED `path` list of the nodes currently on the stack. When you hit a back-edge (id is in `on_stack`), slice `path` from the position of the back-edge target onwards, and append the target again to make the cycle explicit:

```python
def visit(node):
    nid = id(node)
    if nid in perm: return None
    if nid in on_stack:
        i = next(j for j, n in enumerate(path) if id(n) == nid)
        return path[i:] + [node]
    on_stack.add(nid); path.append(node)
    for child in get_children(node):
        cyc = visit(child)
        if cyc is not None: return cyc
    on_stack.discard(nid); path.pop()
    perm.add(nid)
    return None
```

Sibling-subtree contamination — `on_stack.discard` AND `path.pop` must both run on the way back up — otherwise sibling subtrees will (wrongly) see leftover nodes and false-positive.

**Return type.** `list[node] | None`. Don't raise.

In [ ]:
def find_cycle(root, get_children):
    path = []           # nodes currently on the DFS stack, in order
    on_stack = set()    # ids of those nodes — O(1) lookup
    perm = set()        # finished subtrees — safe to skip

    def visit(node):
        nid = id(node)
        if nid in perm:
            return None            # already finished
        if nid in on_stack:
            # back-edge — locate the first appearance of nid in path,
            # slice forward, and CLOSE the loop by repeating the target.
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            return path[i:] + [node]
        on_stack.add(nid)
        path.append(node)
        for child in get_children(node):
            cyc = visit(child)
            if cyc is not None:
                return cyc        # propagate up — found, no further work
        on_stack.discard(nid)     # leaving subtree — drop from stack-tracking
        path.pop()                # ...and from the ordered path
        perm.add(nid)
        return None

    return visit(root)


<details><summary>Solution</summary>

```python
def find_cycle(root, get_children):
    path = []           # nodes currently on the DFS stack, in order
    on_stack = set()    # ids of those nodes — O(1) lookup
    perm = set()        # finished subtrees — safe to skip

    def visit(node):
        nid = id(node)
        if nid in perm:
            return None            # already finished
        if nid in on_stack:
            # back-edge — locate the first appearance of nid in path,
            # slice forward, and CLOSE the loop by repeating the target.
            i = next(j for j, n in enumerate(path) if id(n) == nid)
            return path[i:] + [node]
        on_stack.add(nid)
        path.append(node)
        for child in get_children(node):
            cyc = visit(child)
            if cyc is not None:
                return cyc        # propagate up — found, no further work
        on_stack.discard(nid)     # leaving subtree — drop from stack-tracking
        path.pop()                # ...and from the ordered path
        perm.add(nid)
        return None

    return visit(root)
```

**`path` is the ordered companion of `on_stack`.** `on_stack` gives O(1) membership; `path` preserves the visit order so we can slice from the back-edge target. Both update in lock-step — add+append on enter, discard+pop on exit. The `next(j ... if id(n) == nid)` scan is at worst O(depth), which is what you pay once on the cycle-found path; the DAG case never runs it.

**Why repeat the closing node.** Returning `[a, b]` for an `a → b → a` cycle is ambiguous — is it a 2-cycle or a 1-cycle? Repeating the target as `[a, b, a]` makes the closure explicit and matches how graph-theory texts print cycles.

**Vs. the sibling `dfs-three-set-toposort` atom (ex1).** Same temp-set machinery; that atom RAISES on the back-edge. This one needs the path BEFORE raising — so we collect it and return it. The caller decides whether to `raise ValueError(' -> '.join(repr(n) for n in cyc))` or just log it.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()